# 15 — Explainable AI

Random Forest impurity importance is always available for the classical model. SHAP and CNN Grad-CAM run only if their optional dependencies and saved CNN checkpoints exist. Explanations are descriptive, not causal or clinical evidence.

In [ ]:
import sys
from pathlib import Path
ROOT=Path.cwd().resolve()
if ROOT.name=='post_05': ROOT=ROOT.parent.parent
elif ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import joblib,pandas as pd, matplotlib.pyplot as plt
from src.post_05.common import FEATURES_DIR, MODELS_DIR, FIGURES_DIR, ensure_output_dirs, new_artifact_path, save_csv_new
from src.post_05.explainability import random_forest_importance
ensure_output_dirs()
features=pd.read_csv(FEATURES_DIR/'selected_feature_list.csv').feature.tolist()
rf_path=MODELS_DIR/'random_forest.joblib'
if not rf_path.exists(): print('Run Notebook 09 first; no Random Forest model found.')
else:
    model=joblib.load(rf_path); importance=random_forest_importance(model,features); display(importance.head(25))
    save_csv_new(importance,FEATURES_DIR/'random_forest_explainability_importance.csv')
    ax=importance.head(20).sort_values('importance').plot.barh(x='feature',y='importance',legend=False,figsize=(8,7)); ax.set_title('Random Forest feature importance'); plt.tight_layout(); plt.savefig(new_artifact_path(FIGURES_DIR/'15_random_forest_importance.png'),dpi=160); plt.show()
    try:
        sample=pd.read_csv(FEATURES_DIR/'selected_cycle_features.csv').query("split == 'train'")[features].sample(min(100, len(pd.read_csv(FEATURES_DIR/'selected_cycle_features.csv').query("split == 'train'"))),random_state=42)
        explanation,shap=__import__('src.post_05.explainability',fromlist=['shap_values']).shap_values(model,sample,sample)
        shap.summary_plot(explanation, sample, show=False); plt.savefig(new_artifact_path(FIGURES_DIR/'15_shap_summary.png'),dpi=160,bbox_inches='tight'); plt.show()
    except (ImportError,RuntimeError) as exc: print('Optional SHAP explanation skipped:',exc)
cnn_path=MODELS_DIR/'cnn_multifeature.pt'
if cnn_path.exists(): print('CNN checkpoint is available. Grad-CAM can be run with PyTorch; install optional requirements if needed.')
else: print('CNN Grad-CAM skipped: no multi-feature CNN checkpoint found.')